In [ ]:
import polars as pl
import numpy as np
import math
from collections import Counter
from urllib.parse import urlparse
from sklearn.feature_extraction.text import HashingVectorizer
import re

INPUT_FILE = "../data/merged_final_2.parquet"
OUTPUT_FILE = "../weights/url_embeddings.npy"
URL_COL = "domain"

N_HASH_FEATURES = 30

ADULT_KEYWORDS = {'porn', 'xxx', 'sex', 'adult', 'nude', 'cams', 'erotic', 'hentai', 'milf', 'boobs', 'pussy', 'ass', 'gay', 'lesbian', 'fuck', 'anal', 'cum', 'cunt', 'dick', 'cock', 'cam'}
ILLEGAL_KEYWORDS = {'bet', 'casino', 'poker', 'slot', 'gamble', 'wager', 'odds', 'bingo', 'roulette', 'blackjack', 'dice', 'lottery', 'jackpot', 'vegas', 'bookie', 'sportsbook', 'betting'}

def calculate_entropy(s: str) -> float:
    if not s:
        return 0.0
    counts = Counter(s)
    length = len(s)
    probs = [c / length for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

def extract_url_components(url: str):
    try:
        parsed = urlparse(url.lower())
        domain = parsed.netloc.split(':')[0]
        path = parsed.path
        query = parsed.query
        fragment = parsed.fragment
        return domain, path, query, fragment
    except:
        return "", "", "", ""

def extract_enhanced_features(url: str):
    domain, path, query, fragment = extract_url_components(url)
    full_text = f"{domain} {path} {query} {fragment}"
    
    adult_score = sum(1 for keyword in ADULT_KEYWORDS if keyword in full_text)
    illegal_score = sum(1 for keyword in ILLEGAL_KEYWORDS if keyword in full_text)
    
    domain_parts = domain.split('.')
    tld = domain_parts[-1] if domain_parts else ""
    domain_name = domain_parts[-2] if len(domain_parts) > 1 else ""
    
    has_adult_tld = 1 if tld in {'xxx', 'sex', 'porn', 'adult', 'cam'} else 0
    has_gambling_tld = 1 if tld in {'bet', 'casino', 'poker', 'bingo'} else 0
    
    path_depth = len([p for p in path.split('/') if p])
    query_params = len([p for p in query.split('&') if p])
    
    return [
        adult_score,
        illegal_score,
        has_adult_tld,
        has_gambling_tld,
        path_depth,
        query_params,
        len(domain_name),
        int('https' in url),
        int('http' in url and 'https' not in url),
        int(any(char.isdigit() for char in domain_name)),
        int('-' in domain_name),
        int('_' in domain_name),
        int('.' in domain_name),
        int(any(keyword in domain_name for keyword in ADULT_KEYWORDS)),
        int(any(keyword in domain_name for keyword in ILLEGAL_KEYWORDS))
    ]

df = pl.read_parquet(INPUT_FILE)

special_chars = ['.', '-', '@', '?', '&', '=', '_', '%', '+', '#', '$', '!']

stats_df = df.select([
    pl.col(URL_COL).str.len_bytes().alias("len_url"),
    (pl.col(URL_COL).str.count_matches(r"\d") / (pl.col(URL_COL).str.len_bytes() + 1)).alias("digit_ratio"),
    *[pl.col(URL_COL).str.count_matches(re.escape(c)).alias(f"count_{c}") for c in special_chars],
    pl.col(URL_COL).str.starts_with("https").cast(pl.UInt8).alias("is_https"),
    pl.col(URL_COL).str.contains(r"^(http://|https://)?\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}").cast(pl.UInt8).alias("is_ip"),
    pl.col(URL_COL).str.count_matches(r"[a-zA-Z]").alias("letter_count"),
    (pl.col(URL_COL).str.count_matches(r"[a-zA-Z]") / (pl.col(URL_COL).str.len_bytes() + 1)).alias("letter_ratio")
])

entropy_series = df[URL_COL].map_elements(calculate_entropy, return_dtype=pl.Float64).alias("entropy")

enhanced_features = np.array([extract_enhanced_features(url) for url in df[URL_COL].to_list()])
enhanced_df = pl.DataFrame(enhanced_features, schema=[
    "adult_score", "illegal_score", "has_adult_tld", "has_gambling_tld", 
    "path_depth", "query_params", "domain_name_len", "is_https_full",
    "is_http_only", "domain_has_digits", "domain_has_hyphen", 
    "domain_has_underscore", "domain_has_dot", "domain_adult_kw", "domain_illegal_kw"
])

path_text_list = []
for url in df[URL_COL].to_list():
    domain, path, query, fragment = extract_url_components(url)
    keyword_context = " ".join([
        "adult" if any(kw in domain for kw in ADULT_KEYWORDS) else "",
        "illegal" if any(kw in domain for kw in ILLEGAL_KEYWORDS) else "",
        path.replace('/', ' '), 
        query.replace('&', ' '), 
        fragment
    ])
    cleaned = re.sub(r'[^a-zA-Z0-9]', ' ', keyword_context + ' ' + domain)
    path_text_list.append(re.sub(r'\s+', ' ', cleaned).strip())

vectorizer = HashingVectorizer(
    n_features=N_HASH_FEATURES,
    alternate_sign=False,
    norm='l2',
    token_pattern=r'(?u)\b\w{2,}\b',
    stop_words=['com', 'www', 'http', 'https', 'org', 'net', 'info', 'co', 'uk', 'html', 'php', 'asp', 'jsp']
)
hashed_features = vectorizer.transform(path_text_list).toarray()

stats_numpy = stats_df.to_numpy()
entropy_numpy = entropy_series.to_numpy().reshape(-1, 1)
enhanced_numpy = enhanced_df.to_numpy()

final_embedding = np.hstack([stats_numpy, entropy_numpy, enhanced_numpy, hashed_features])
final_embedding = final_embedding.astype(np.float32)

print(f"Final Embedding Shape: {final_embedding.shape}")
np.save(OUTPUT_FILE, final_embedding)

Final Embedding Shape: (19405, 64)


In [2]:
import polars as pl
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler

df = pl.read_parquet("../data/merged_final_2.parquet")

text_data = df['text'].fill_null('').to_list()
domain_data = df['domain'].fill_null('').to_list()

combined_text = [f"{t} {d}" for t, d in zip(text_data, domain_data)]

count_vectorizer = CountVectorizer(
    max_features=10000,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

count_embeddings = count_vectorizer.fit_transform(combined_text).toarray().astype(np.float32)
tfidf_embeddings = tfidf_vectorizer.fit_transform(combined_text).toarray().astype(np.float32)

np.save('../weights/count_vectorizer_embeddings.npy', count_embeddings)
np.save('../weights/tfidf_vectorizer_embeddings.npy', tfidf_embeddings)